# 7. Part D - experiment 1: Inspect the Sparse Representation

In [5]:
import json
from implementation import (
    compute_counts,
    compute_tf,
    compute_idf,
    compute_tfidf,
)
from collections import Counter
# raw documents
documents = []
path = "/home/cong/26_8_learn_rag/NLP/" + "c4-train.00000-of-01024-30K.json"
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        doc = json.loads(line)
        text = doc["text"].lower()
        documents.append(text)
print("Number of documents: ", len(documents))
# documents = documents[:1000]

Number of documents:  30000


In [6]:
# function build vocab
def build_vocabulary(
    documents,
    min_frequency=1,
    max_frequency_ratio=1.0
):
    """
    Build vocabulary from a collection of documents.

    Parameters
    ----------
    documents : list[str]
        List of raw text documents.

    min_frequency : int
        Minimum total number of times a term must appear
        in the whole corpus.

    max_frequency_ratio : float
        Maximum ratio of documents containing a term.

    Returns
    -------
    list[str]
        Sorted vocabulary.
    """

    term_frequency = Counter()
    document_frequency = Counter()

    for doc in documents:

        # Tokenizer
        tokens = re.findall(
            r"\b[a-zA-Z]+\b",
            doc.lower()
        )

        # Total frequency in the corpus
        term_frequency.update(tokens)

        # Number of documents containing the term
        unique_tokens = set(tokens)
        document_frequency.update(unique_tokens)

    num_documents = len(documents)

    vocabulary = []

    for word, frequency in term_frequency.items():

        # Remove rare terms
        if frequency < min_frequency:
            continue

        # Remove terms appearing in too many documents
        df = document_frequency[word]

        if df / num_documents > max_frequency_ratio:
            continue

        vocabulary.append(word)

    return sorted(vocabulary)

In [7]:
# tokenizer
import re
def tokenize(text):
    return re.findall(r"\b[a-zA-Z]+\b", text.lower())

tokenized_documents = []

for doc in documents:
    tokens = tokenize(doc)
    tokenized_documents.append(tokens)

print("Example tokens: ", tokenized_documents[0][:20])

Example tokens:  ['beginners', 'bbq', 'class', 'taking', 'place', 'in', 'missoula', 'do', 'you', 'want', 'to', 'get', 'better', 'at', 'making', 'delicious', 'bbq', 'you', 'will', 'have']


In [8]:
# count vectorizer
tokenized_texts = [" ".join(tokens) for tokens in tokenized_documents]

vocab = build_vocabulary(
    tokenized_texts,
    min_frequency=5
)

print("Vocab_size:", len(vocab))
print("First elements in vocab", vocab[:100])

Vocab_size: 49824
First elements in vocab ['a', 'aa', 'aaa', 'aaba', 'aac', 'aace', 'aadhaar', 'aadhar', 'aadvantage', 'aaem', 'aah', 'aaldef', 'aalto', 'aamir', 'aan', 'aao', 'aap', 'aapl', 'aar', 'aard', 'aarhus', 'aaron', 'aaronsek', 'aarp', 'aarti', 'aas', 'aata', 'ab', 'aba', 'ababa', 'abacityblog', 'aback', 'abacus', 'abadas', 'abandon', 'abandoned', 'abandoning', 'abandonment', 'abandons', 'abarth', 'abarths', 'abated', 'abatement', 'abatements', 'abattoir', 'abaya', 'abb', 'abba', 'abbas', 'abbess', 'abbey', 'abbo', 'abbot', 'abbotsford', 'abbott', 'abbreviated', 'abbreviation', 'abbreviations', 'abby', 'abc', 'abcd', 'abcp', 'abcs', 'abd', 'abdallah', 'abdomen', 'abdominal', 'abdoulaye', 'abducted', 'abduction', 'abdul', 'abdullah', 'abe', 'abel', 'abell', 'abend', 'aber', 'abercrombie', 'aberdeen', 'aberfeldy', 'abergavenny', 'aberration', 'aberrations', 'abet', 'abhishek', 'abhyas', 'abi', 'abide', 'abiding', 'abierto', 'abigail', 'abilene', 'abilities', 'ability', 'abilt', 

In [5]:
counts_list = compute_counts(
    tokenized_texts, 
    vocab
)

print("count matrix shape")
print(len(counts_list), "x", len(vocab))

count matrix shape
1000 x 7154


In [6]:
# tf
tf_list = compute_tf(
    counts_list,
    vocab
)

print("TF matrix shape:")
print(
    len(tf_list),
    "x",
    len(vocab)
)

TF matrix shape:
1000 x 7154


In [7]:
# idf
idf_list = compute_idf(
    counts_list,
    vocab
)
print("IDF vector size: ", len(idf_list))

print("first 5 idf values", idf_list[:5])
tfidf_matrix = compute_tfidf(
    tf_list,
    idf_list,
    vocab
)

print("TF-IDF matrix shape:")
print(
    len(tfidf_matrix),
    "x",
    len(vocab)
)

IDF vector size:  7154
first 5 idf values [0.14734058789870913, 5.809142990314028, 5.115995809754082, 6.907755278982137, 6.907755278982137]
TF-IDF matrix shape:
1000 x 7154


In [8]:
# Check sparsity

N = len(tfidf_matrix)
V = len(vocab)

nnz = 0

for row in tfidf_matrix:
    for value in row:
        if value != 0:
            nnz += 1

total_elements = N * V

sparsity = 1 - nnz / total_elements

print("\n===== SPARSITY =====")
print("Number of documents:", N)
print("Vocabulary size:", V)
print("Total elements:", total_elements)
print("Non-zero elements:", nnz)
print("Sparsity:", sparsity)
print("Sparsity (%):", sparsity * 100)


===== SPARSITY =====
Number of documents: 1000
Vocabulary size: 7154
Total elements: 7154000
Non-zero elements: 142486
Sparsity: 0.9800830304724629
Sparsity (%): 98.00830304724629


In [9]:
# ============================================================
# 7.5. Inspect vocabulary
# ============================================================

# ------------------------------------------------------------
# 7.5.1. 20 terms phổ biến nhất theo Document Frequency
# ------------------------------------------------------------

df_list = []

for idx, word in enumerate(vocab):

    df = 0

    for doc_counts in counts_list:
        if doc_counts[idx] > 0:
            df += 1

    df_list.append(df)


df_ranking = sorted(
    zip(vocab, df_list),
    key=lambda x: x[1],
    reverse=True
)

print("\n===== 20 TERMS - HIGHEST DOCUMENT FREQUENCY =====")

for word, df in df_ranking[:20]:
    print(
        f"{word:<20} DF = {df}"
    )


# ------------------------------------------------------------
# 7.5.2. 20 terms có IDF cao nhất
# ------------------------------------------------------------

idf_ranking = sorted(
    zip(vocab, idf_list),
    key=lambda x: x[1],
    reverse=True
)

print("\n===== 20 TERMS - HIGHEST IDF =====")

for word, idf in idf_ranking[:20]:
    print(
        f"{word:<20} IDF = {idf:.6f}"
    )


# ------------------------------------------------------------
# 7.5.3. 20 terms có TF-IDF cao nhất
# trong document được chọn
# ------------------------------------------------------------

document_idx = 20

tfidf_ranking = []

for idx, value in enumerate(
    tfidf_matrix[document_idx]
):
    if value != 0:
        tfidf_ranking.append(
            (vocab[idx], value)
        )


tfidf_ranking.sort(
    key=lambda x: x[1],
    reverse=True
)

print(
    "\n===== 20 TERMS - HIGHEST TF-IDF "
    f"IN DOCUMENT {document_idx} ====="
)

for word, tfidf in tfidf_ranking[:20]:
    print(
        f"{word:<20} TF-IDF = {tfidf:.6f}"
    )


# ============================================================
# 7.5.4. So sánh ba danh sách
# ============================================================

top_df_terms = {
    word for word, df in df_ranking[:20]
}

top_idf_terms = {
    word for word, idf in idf_ranking[:20]
}

top_tfidf_terms = {
    word for word, tfidf in tfidf_ranking[:20]
}


print("\n===== COMPARISON =====")

print(
    "DF ∩ IDF:",
    top_df_terms & top_idf_terms
)

print(
    "DF ∩ TF-IDF:",
    top_df_terms & top_tfidf_terms
)

print(
    "IDF ∩ TF-IDF:",
    top_idf_terms & top_tfidf_terms
)

print(
    "DF ∩ IDF ∩ TF-IDF:",
    (
        top_df_terms
        & top_idf_terms
        & top_tfidf_terms
    )
)


===== 20 TERMS - HIGHEST DOCUMENT FREQUENCY =====
the                  DF = 939
and                  DF = 919
to                   DF = 896
of                   DF = 887
a                    DF = 863
in                   DF = 855
for                  DF = 772
is                   DF = 755
with                 DF = 734
on                   DF = 684
that                 DF = 632
this                 DF = 595
s                    DF = 594
are                  DF = 587
it                   DF = 583
be                   DF = 573
from                 DF = 569
as                   DF = 565
at                   DF = 551
you                  DF = 541

===== 20 TERMS - HIGHEST IDF =====
ab                   IDF = 6.907755
abbo                 IDF = 6.907755
abc                  IDF = 6.907755
abrasion             IDF = 6.907755
acceleration         IDF = 6.907755
acer                 IDF = 6.907755
adast                IDF = 6.907755
addon                IDF = 6.907755
adenoma              IDF 

## Trả lời:
* Do ở đây kích thước documents khá lớn (em chạy bị died kernel) nên em lấy khoảng 1000 documents
* Vocab_size = 7154 của 1000 documents
* Matrix shape = 1000 * 7154
---
* N = 1000
* V = 7154
---
Tại sao một document chỉ sử dụng một phần rất nhỏ vocabulary nhưng vector vẫn có chiều (V)?
* Vì mỗi chiều của vector tương ứng với một term trong toàn bộ vocabulary của corpus. Nếu vocabulary V terms thì mọi documents đều được biểu diễn trong cùng 1 không gian V chiều để có thể so sánh giữa các document
* Document không chứa term nào thì giá trị TF-IDF của term đó bằng 0

---
Một term xuất hiện rất nhiều trong corpus có nhất thiết có TF-IDF cao không?
* Một term xuất hiện nhiều lần trong document thì có khả năng TF-IDF vẫn thấp vì IDF có thể thấp

Một term có IDF cao có nhất thiết có TF-IDF cao trong mọi document không?
* Không vì có thể TF sẽ thấp

# Build vocab from 30000 documents

In [10]:
from implementation import build_vocabulary, compute_counts, compute_tf, compute_idf, compute_tfidf, cosine_similarity
print("Import sucessfully")

Import sucessfully


In [11]:
corpus = [
    "cat eats fish", "dog eats fish", "cat likes fish"
]

In [12]:
import math


def run_unit_tests():
    corpus = [
        "cat eats fish",
        "dog eats fish",
        "cat likes fish"
    ]
    vocab = build_vocabulary(corpus)

    expected_vocab = [
        "cat",
        "dog",
        "eats",
        "fish",
        "likes"
    ]

    assert vocab == expected_vocab
    print("Test build_vocabulary: ĐẠT")
    counts_list = compute_counts(
        corpus,
        vocab
    )

    expected_counts = [
        [1, 0, 1, 1, 0],
        [0, 1, 1, 1, 0],
        [1, 0, 0, 1, 1]
    ]

    assert counts_list == expected_counts
    print("Test compute_counts: ĐẠT")

    tf_list = compute_tf(
        counts_list,
        vocab
    )

    tf_cat_d1 = tf_list[0][0]

    assert abs(
        tf_cat_d1 - (1 / 3)
    ) < 1e-9

    tf_dog_d1 = tf_list[0][1]

    assert abs(
        tf_dog_d1 - 0
    ) < 1e-9

    tf_dog_d2 = tf_list[1][1]

    assert abs(
        tf_dog_d2 - (1 / 3)
    ) < 1e-9

    print("Test compute_tf: ĐẠT")

    idf_list = compute_idf(
        counts_list,
        vocab
    )

    idf_cat = idf_list[0]

    assert abs(
        idf_cat - math.log(3 / 2)
    ) < 1e-9

    idf_dog = idf_list[1]

    assert abs(
        idf_dog - math.log(3)
    ) < 1e-9

    idf_fish = idf_list[3]

    assert abs(
        idf_fish - 0
    ) < 1e-9

    print("Test compute_idf: ĐẠT")

    tfidf_matrix = compute_tfidf(
        tf_list,
        idf_list,
        vocab
    )

    tfidf_cat_d1 = tfidf_matrix[0][0]

    expected_tfidf_cat_d1 = (
        (1 / 3) * math.log(3 / 2)
    )

    assert abs(
        tfidf_cat_d1 - expected_tfidf_cat_d1
    ) < 1e-9

    tfidf_dog_d2 = tfidf_matrix[1][1]

    expected_tfidf_dog_d2 = (
        (1 / 3) * math.log(3)
    )

    assert abs(
        tfidf_dog_d2 - expected_tfidf_dog_d2
    ) < 1e-9

    tfidf_fish_d1 = tfidf_matrix[0][3]

    assert abs(
        tfidf_fish_d1 - 0
    ) < 1e-9

    print("Test compute_tfidf: ĐẠT")

    print("\nTẤT CẢ UNIT TEST: ĐẠT")

run_unit_tests()

Test build_vocabulary: ĐẠT
Test compute_counts: ĐẠT
Test compute_tf: ĐẠT
Test compute_idf: ĐẠT
Test compute_tfidf: ĐẠT

TẤT CẢ UNIT TEST: ĐẠT


In [13]:
import math

from sklearn.feature_extraction.text import TfidfVectorizer


def compare_vocabulary(student_vocab, reference_vocab):
    print("\n========== VOCABULARY ==========")

    print("Student vocabulary:")
    print(student_vocab)

    print("\nReference vocabulary:")
    print(reference_vocab)

    if student_vocab == reference_vocab:
        print("\nVocabulary: GIỐNG NHAU")
        return True

    print("\nVocabulary: KHÁC NHAU")

    student_set = set(student_vocab)
    reference_set = set(reference_vocab)

    only_student = student_set - reference_set
    only_reference = reference_set - student_set

    print("\nChỉ có trong Student:")
    print(sorted(only_student))

    print("\nChỉ có trong Reference:")
    print(sorted(only_reference))

    return False


def compare_tfidf(
    student_matrix,
    reference_matrix,
    student_vocab,
    reference_vocab,
    tolerance=1e-9
):
    print("\n========== TF-IDF ==========")

    if student_vocab != reference_vocab:
        print(
            "Vocabulary ordering khác nhau, "
            "không thể so sánh trực tiếp từng cột."
        )
        return False

    if len(student_matrix) != len(reference_matrix):
        print("Số lượng document khác nhau.")
        return False

    for i in range(len(student_matrix)):
        if len(student_matrix[i]) != len(reference_matrix[i]):
            print(
                f"Số lượng feature tại document {i} "
                "khác nhau."
            )
            return False

    max_difference = 0.0
    different_values = []

    for i in range(len(student_matrix)):
        for j in range(len(student_matrix[i])):
            student_value = student_matrix[i][j]
            reference_value = reference_matrix[i][j]

            difference = abs(
                student_value - reference_value
            )

            max_difference = max(
                max_difference,
                difference
            )

            if difference > tolerance:
                different_values.append(
                    (
                        i,
                        j,
                        student_value,
                        reference_value,
                        difference
                    )
                )

    print(
        f"Maximum absolute difference: "
        f"{max_difference}"
    )

    if not different_values:
        print("TF-IDF: GIỐNG NHAU")
        return True

    print(
        f"TF-IDF: KHÁC NHAU "
        f"({len(different_values)} values)"
    )

    print("\nMột số giá trị khác nhau:")

    for item in different_values[:10]:
        i, j, student, reference, diff = item

        print(
            f"Document={i}, "
            f"Feature={j}, "
            f"Student={student:.12f}, "
            f"Reference={reference:.12f}, "
            f"Diff={diff:.12f}"
        )

    return False


def run_reference_comparison():
    corpus = [
        "cat eats fish",
        "dog eats fish",
        "cat likes fish"
    ]

    # ==================================================
    # 1. Student implementation
    # ==================================================

    student_vocab = build_vocabulary(corpus)

    student_counts = compute_counts(
        corpus,
        student_vocab
    )

    student_tf = compute_tf(
        student_counts,
        student_vocab
    )

    student_idf = compute_idf(
        student_counts,
        student_vocab
    )

    student_tfidf = compute_tfidf(
        student_tf,
        student_idf,
        student_vocab
    )

    # ==================================================
    # 2. Reference implementation
    # ==================================================

    vectorizer = TfidfVectorizer(
        lowercase=True,
        token_pattern=r"(?u)\b\w+\b",
        norm=None,
        use_idf=True,
        smooth_idf=False
    )

    reference_matrix = vectorizer.fit_transform(
        corpus
    )

    reference_vocab_dict = (
        vectorizer.vocabulary_
    )

    reference_vocab = [
        word
        for word, index
        in sorted(
            reference_vocab_dict.items(),
            key=lambda x: x[1]
        )
    ]

    reference_matrix = (
        reference_matrix
        .toarray()
        .tolist()
    )

    # ==================================================
    # 3. Compare vocabulary
    # ==================================================

    vocab_same = compare_vocabulary(
        student_vocab,
        reference_vocab
    )

    # ==================================================
    # 4. Compare TF-IDF
    # ==================================================

    tfidf_same = compare_tfidf(
        student_tfidf,
        reference_matrix,
        student_vocab,
        reference_vocab
    )

    # ==================================================
    # 5. Final result
    # ==================================================

    print("\n========== SUMMARY ==========")

    print(
        f"Vocabulary: "
        f"{'MATCH' if vocab_same else 'DIFFER'}"
    )

    print(
        f"TF-IDF: "
        f"{'MATCH' if tfidf_same else 'DIFFER'}"
    )

    if vocab_same and tfidf_same:
        print(
            "\nStudent implementation và "
            "Reference implementation: GIỐNG NHAU"
        )
    else:
        print(
            "\nStudent implementation và "
            "Reference implementation: KHÁC NHAU"
        )

run_reference_comparison()


========== VOCABULARY ==========
Student vocabulary:
['cat', 'dog', 'eats', 'fish', 'likes']

Reference vocabulary:
['cat', 'dog', 'eats', 'fish', 'likes']

Vocabulary: GIỐNG NHAU

========== TF-IDF ==========
Maximum absolute difference: 1.7324081924454067
TF-IDF: KHÁC NHAU (9 values)

Một số giá trị khác nhau:
Document=0, Feature=0, Student=0.135155036036, Reference=1.405465108108, Diff=1.270310072072
Document=0, Feature=2, Student=0.135155036036, Reference=1.405465108108, Diff=1.270310072072
Document=0, Feature=3, Student=0.000000000000, Reference=1.000000000000, Diff=1.000000000000
Document=1, Feature=1, Student=0.366204096223, Reference=2.098612288668, Diff=1.732408192445
Document=1, Feature=2, Student=0.135155036036, Reference=1.405465108108, Diff=1.270310072072
Document=1, Feature=3, Student=0.000000000000, Reference=1.000000000000, Diff=1.000000000000
Document=2, Feature=0, Student=0.135155036036, Reference=1.405465108108, Diff=1.270310072072
Document=2, Feature=3, Student=0.0

## Trả lời

Ở đây có sự khác biệt giữa Student implementation và Reference implementation do hai bên sử dụng **quy ước tính IDF khác nhau**.

Student implementation sử dụng:

\[
IDF(t)=\log\frac{N}{df(t)}
\]

Trong khi Reference implementation của thư viện sử dụng:

\[
IDF(t)=\log\frac{N}{df(t)}+1
\]

Do đó, giá trị IDF và TF-IDF giữa hai implementation có sự chênh lệch. Đây là sự khác biệt về **convention trong công thức IDF**.

# Part F - Experiment 2: Preprocessing Ablation

In [14]:
# function build vocab
def build_vocabulary(
    documents,
    min_frequency=1,
    max_frequency_ratio=1.0
):
    """
    Build vocabulary from a collection of documents.

    Parameters
    ----------
    documents : list[str]
        List of raw text documents.

    min_frequency : int
        Minimum total number of times a term must appear
        in the whole corpus.

    max_frequency_ratio : float
        Maximum ratio of documents containing a term.

    Returns
    -------
    list[str]
        Sorted vocabulary.
    """

    term_frequency = Counter()
    document_frequency = Counter()

    for doc in documents:

        # Tokenizer
        tokens = re.findall(
            r"\b[a-zA-Z]+\b",
            doc.lower()
        )

        # Total frequency in the corpus
        term_frequency.update(tokens)

        # Number of documents containing the term
        unique_tokens = set(tokens)
        document_frequency.update(unique_tokens)

    num_documents = len(documents)

    vocabulary = []

    for word, frequency in term_frequency.items():

        # Remove rare terms
        if frequency < min_frequency:
            continue

        # Remove terms appearing in too many documents
        df = document_frequency[word]

        if df / num_documents > max_frequency_ratio:
            continue

        vocabulary.append(word)

    return sorted(vocabulary)

In [15]:
# pipeline A
import re

def preprocess_A(text):
    text = text.lower()

    tokens = re.findall(
        r"\b[a-zA-Z]+\b",
        text
    )

    return tokens

In [16]:
# pipeline B
import re

STOPWORDS = {
    "the", "a", "an", "and", "or",
    "is", "are", "was", "were", "to",
    "of", "in", "on", "for", "with"
}

def preprocess_B(text):
    text = text.lower()

    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    tokens = re.findall(
        r"\b[a-zA-Z]+\b",
        text
    )

    tokens = [token for token in tokens if token not in STOPWORDS]

    return tokens

In [17]:
# pipeline C
def preprocess_C(text):
    text = text.lower()

    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    words = re.findall(
        r"\b[a-zA-Z]+\b",
        text
    )

    subwords = []

    for word in words:
        if len(word) <= 4:
            subwords.append(word)
        else:
            subwords.append(word[:4])
            subwords.append(
                "##" + word[4:]
            )
    return subwords

In [18]:
# run pipeline 
def run_pipeline(documents, preprocess_fn, min_frequency=5):

    tokenized_docs = []

    for doc in documents:
        tokenized_docs.append(
            preprocess_fn(doc)
        )

    tokenized_texts = [
        " ".join(tokens)
        for tokens in tokenized_docs
    ]

    vocab = build_vocabulary(
        tokenized_texts,
        min_frequency=min_frequency
    )

    counts = compute_counts(
        tokenized_texts,
        vocab
    )

    tf = compute_tf(
        counts,
        vocab
    )

    idf = compute_idf(
        counts,
        vocab
    )

    tfidf = compute_tfidf(
        tf,
        idf,
        vocab
    )

    return {
        "vocab": vocab,
        "counts": counts,
        "tf": tf,
        "idf": idf,
        "tfidf": tfidf,
        "tokenized_docs": tokenized_docs
    }

In [19]:
# vocab size
def vocabulary_size(result):
    return len(result["vocab"])

# average tokens per document
def average_tokens(result):

    docs = result["tokenized_docs"]

    total = sum(
        len(doc)
        for doc in docs
    )

    return total / len(docs)

# matrix sparsity
def compute_sparsity(tfidf_matrix):

    if not tfidf_matrix or not tfidf_matrix[0]:
        return 0.0

    N = len(tfidf_matrix)
    V = len(tfidf_matrix[0])

    nnz = 0

    for row in tfidf_matrix:
        for value in row:
            if value != 0:
                nnz += 1

    return 1 - (
        nnz / (N * V)
)

In [20]:
train_docs = documents[:25000]
test_docs = documents[25000:]

In [21]:
def compute_oov_rate(
    train_vocab,
    test_docs,
    preprocess_fn
):

    total = 0
    oov = 0

    vocab_set = set(train_vocab)

    for doc in test_docs:

        tokens = preprocess_fn(doc)

        for token in tokens:

            total += 1

            if token not in vocab_set:
                oov += 1

    return oov / total

In [22]:
result_A = run_pipeline(
    documents,
    preprocess_A
)

result_B = run_pipeline(
    documents,
    preprocess_B
)

result_C = run_pipeline(
    documents,
    preprocess_C
)

In [23]:
print(
    "Pipeline | Vocab | Avg Tokens | Sparsity"
)

for name, result in [
    ("A", result_A),
    ("B", result_B),
    ("C", result_C)
]:

    print(
        name,
        vocabulary_size(result),
        average_tokens(result),
        compute_sparsity(
            result["tfidf"]
        )
    )

Pipeline | Vocab | Avg Tokens | Sparsity
A 7154 371.742 0.9800830304724629
B 7174 286.348 0.9815129634792306
C 6497 531.282 0.9779110358627059


# 10. Part G - Application: Build a Document Search Engine

In [ ]:
import re

def build_query_vector(query, vocab, idf_list):
    tokens = re.findall(
        r"\b[a-zA-Z]+\b",
        query.lower()
    )

    word_counts = {}
    
    for token in tokens:
        word_counts[token] = (
            word_counts.get(token, 0) + 1
        )

    total_words = len(tokens)
    tfidf_vector = []

    for idx, word in enumerate(vocab):
        count = word_counts.get(word, 0)

        tf = (
            count / total_words if total_words > 0 else 0
        )

        tfidf = tf * idf_list[idx]

        tfidf_vector.append(tfidf)

    return tfidf_vector

In [26]:
# function to search vector
def search_documents(query, documents, vocab, idf_list, tfidf_matrix, top_k = 5):
    query_vector = build_query_vector(query, vocab, idf_list)

    scores = []

    for doc_id, doc_vector in enumerate(tfidf_matrix):
        similarity = cosine_similarity(query_vector, doc_vector)
    
        scores.append((doc_id, similarity))

    scores.sort(key=lambda x : x[1], reverse=True)

    return scores[:top_k]

In [27]:
# function to show results
def show_results(query, results, documents):
    print("\n" + "-"*80)
    print("QUERY: ", query)

    print("TOP RESULTS")
    
    print(
        f"{'Rank':<5}"
        f"{'Doc ID':<10}"
        f"{'Similarity':<15}"
        f"{'Preview'}"
    )

    for rank, (doc_id, score) in enumerate(
        results,
        start=1
    ):

        preview = (
            documents[doc_id]
            .replace("\n", " ")
            [:120]
        )

        print(
            f"{rank:<5}"
            f"{doc_id:<10}"
            f"{score:<15.6f}"
            f"{preview}"
        )

In [28]:
queries = [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing"
]

for query in queries:

    results = search_documents(
        query=query,
        documents=documents,
        vocab=vocab,
        idf_list=idf_list,
        tfidf_matrix=tfidf_matrix,
        top_k=5
    )

    show_results(
        query,
        results,
        documents
    )


--------------------------------------------------------------------------------
QUERY:  medical image classification
TOP RESULTS
Rank Doc ID    Similarity     Preview
1    190       0.307839       this title is a comprehensive account of the key aspects of medical leadership. a highly accessible, text book-style res
2    207       0.290287       rent the centre. celebrations, parties or just for fun. in order to provide a flavour of the grenville house experience 
3    267       0.191922       image title: pottery barn counter stools in decker leather seat barstool inside bar plans 8. filename: pottery-barn-coun
4    515       0.188356       wind river opens in theaters august 4, 2017, courtesy of the weinstein company. top image photographed by sandro baebler
5    871       0.169593       so you want a college degree. in the usa, about the only acceptable answer is “chaaa, to get a bitchin’ new camaro, homb

---------------------------------------------------------------------------

# 11. Part H — Evaluation

In [ ]:
# 11.1. Tạo evaluation set gồm các queries và relevance labels (ground-truth)
evaluation_set = [
    {
        "query": "medical image classification",
        "relevant_docs": {190, 267, 515, 793}
    },
    {
        "query": "transformer language model",
        "relevant_docs": {701, 700, 350, 582}
    },
    {
        "query": "deep learning healthcare",
        "relevant_docs": {838, 575, 196, 454}
    },
    {
        "query": "natural language processing",
        "relevant_docs": {701, 700, 350, 582}
    },
    {
        "query": "database sql query",
        "relevant_docs": {408, 757, 768, 914}
    },
    {
        "query": "climate change global warming",
        "relevant_docs": {896, 80}
    }
]

print("===== EVALUATION BENCHMARK =====")
print(f"Tổng số queries: {len(evaluation_set)}\n")
for i, item in enumerate(evaluation_set, start=1):
    print(f"Query {i}: \"{item['query']}\"")    
    print(f"  -> Relevant Doc IDs: {sorted(list(item['relevant_docs']))}")


===== EVALUATION BENCHMARK =====
Tổng số queries: 6

Query 1: "medical image classification"
  -> Relevant Doc IDs: [190, 267, 515, 793]
Query 2: "transformer language model"
  -> Relevant Doc IDs: [350, 582, 700, 701]
Query 3: "deep learning healthcare"
  -> Relevant Doc IDs: [196, 454, 575, 838]
Query 4: "natural language processing"
  -> Relevant Doc IDs: [350, 582, 700, 701]
Query 5: "database sql query"
  -> Relevant Doc IDs: [408, 757, 768, 914]
Query 6: "climate change global warming"
  -> Relevant Doc IDs: [80, 896]


In [ ]:
# 11.2, 11.3, 11.4. Định nghĩa các hàm tính Metric: Precision@K, Recall@K, Reciprocal Rank

def compute_precision_at_k(retrieved_doc_ids, relevant_doc_ids, k=5):
    """Nhận xét & Đánh giá kết quả thực nghiệm

    Tính Precision@K:
    P@K = (# relevant documents retrieved in top K) / K
    """
    top_k_retrieved = retrieved_doc_ids[:k]
    relevant_retrieved = [doc_id for doc_id in top_k_retrieved if doc_id in relevant_doc_ids]
    return len(relevant_retrieved) / k


def compute_recall_at_k(retrieved_doc_ids, relevant_doc_ids, k=5):
    """
    Tính Recall@K:
    R@K = (# relevant documents retrieved in top K) / (# relevant documents)
    """
    if not relevant_doc_ids:
        return 0.0
    top_k_retrieved = retrieved_doc_ids[:k]
    relevant_retrieved = [doc_id for doc_id in top_k_retrieved if doc_id in relevant_doc_ids]
    return len(relevant_retrieved) / len(relevant_doc_ids)


def compute_reciprocal_rank(retrieved_doc_ids, relevant_doc_ids):
    """
    Tính Reciprocal Rank (RR):
    RR = 1 / r_q nếu document relevant đầu tiên xuất hiện ở vị trí rank r_q (1-indexed).
    Nếu không tìm thấy document relevant nào trong top retrieved, trả về 0.0.
    """
    for rank, doc_id in enumerate(retrieved_doc_ids, start=1):
        if doc_id in relevant_doc_ids:
            return 1.0 / rank
    return 0.0


In [34]:
# Thực hiện đánh giá trên toàn bộ evaluation set với top_k = 5 và tự động lưu ra file .csv
import csv

k = 5
results_summary = []
csv_rows = []
all_p5 = []
all_r5 = []
all_rr = []

print("=" * 100)
print(f"{'STT':<4} | {'Query':<32} | {'Retrieved Top 5':<26} | {'P@5':<8} | {'R@5':<8} | {'RR':<8}")
print("=" * 100)

for idx, item in enumerate(evaluation_set, start=1):
    query = item["query"]
    relevant_docs = item["relevant_docs"]
    
    # 1. Thực hiện tìm kiếm (Retrieval)
    search_results = search_documents(
        query=query,
        documents=documents,
        vocab=vocab,
        idf_list=idf_list,
        tfidf_matrix=tfidf_matrix,
        top_k=k
    )
    
    retrieved_ids = [doc_id for doc_id, score in search_results]
    
    # 2. Tính toán các chỉ số đánh giá (Evaluation)
    p5 = compute_precision_at_k(retrieved_ids, relevant_docs, k=k)
    r5 = compute_recall_at_k(retrieved_ids, relevant_docs, k=k)
    rr = compute_reciprocal_rank(retrieved_ids, relevant_docs)
    
    all_p5.append(p5)
    all_r5.append(r5)
    all_rr.append(rr)
    
    results_summary.append({
        "stt": idx,
        "query": query,
        "retrieved": retrieved_ids,
        "p5": p5,
        "r5": r5,
        "rr": rr
    })
    
    print(f"{idx:<4} | {query:<32} | {str(retrieved_ids):<26} | {p5:<8.4f} | {r5:<8.4f} | {rr:<8.4f}")
    
    # 3. Thu thập dữ liệu chi tiết cho từng document được retrieve
    for rank, (doc_id, score) in enumerate(search_results, start=1):
        is_relevant = doc_id in relevant_docs
        preview = documents[doc_id].replace("\n", " ").strip()[:120]
        csv_rows.append({
            "Query_ID": idx,
            "Query": query,
            "P@5": f"{p5:.4f}",
            "R@5": f"{r5:.4f}",
            "RR": f"{rr:.4f}",
            "Rank": rank,
            "Doc_ID": doc_id,
            "Similarity": f"{score:.6f}",
            "Is_Relevant": is_relevant,
            "Relevant_Docs": "; ".join(map(str, sorted(list(relevant_docs)))),
            "Preview": preview
        })

# Tính Mean của các metrics
mean_p5 = sum(all_p5) / len(all_p5)
mean_r5 = sum(all_r5) / len(all_r5)
mrr = sum(all_rr) / len(all_rr)

print("=" * 100)
print(f"{'TRUNG BÌNH TOÀN BỘ BENCHMARK (Mean)':<65} | {mean_p5:<8.4f} | {mean_r5:<8.4f} | {mrr:<8.4f}")
print("=" * 100)

# Thêm dòng tổng kết Mean vào cuối file CSV
csv_rows.append({
    "Query_ID": "MEAN",
    "Query": "OVERALL BENCHMARK AVERAGE",
    "P@5": f"{mean_p5:.4f}",
    "R@5": f"{mean_r5:.4f}",
    "RR": f"{mrr:.4f}",
    "Rank": "",
    "Doc_ID": "",
    "Similarity": "",
    "Is_Relevant": "",
    "Relevant_Docs": "",
    "Preview": "Mean evaluation metrics across all benchmark queries"
})

# 4. TỰ ĐỘNG LƯU RA FILE .CSV KHI CHẠY XONG
csv_filename = "results.csv"
fieldnames = [
    "Query_ID", "Query", "P@5", "R@5", "RR",
    "Rank", "Doc_ID", "Similarity", "Is_Relevant",
    "Relevant_Docs", "Preview"
]

with open(csv_filename, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(csv_rows)

print(f"\n[THÀNH CÔNG] Đã tự động lưu {len(csv_rows)} dòng kết quả vào file: {csv_filename}")

STT  | Query                            | Retrieved Top 5            | P@5      | R@5      | RR      
1    | medical image classification     | [190, 207, 267, 515, 871]  | 0.6000   | 0.7500   | 1.0000  
2    | transformer language model       | [701, 700, 350, 582, 914]  | 0.8000   | 1.0000   | 1.0000  
3    | deep learning healthcare         | [838, 575, 208, 820, 196]  | 0.6000   | 0.7500   | 1.0000  
4    | natural language processing      | [701, 700, 350, 565, 582]  | 0.8000   | 1.0000   | 1.0000  
5    | database sql query               | [408, 686, 757, 287, 768]  | 0.6000   | 0.7500   | 1.0000  
6    | climate change global warming    | [896, 291, 80, 110, 757]   | 0.4000   | 1.0000   | 1.0000  
TRUNG BÌNH TOÀN BỘ BENCHMARK (Mean)                               | 0.6333   | 0.8750   | 1.0000  

[THÀNH CÔNG] Đã tự động lưu 31 dòng kết quả vào file: results.csv


## Nhận xét & Đánh giá kết quả thực nghiệm

### 1. Phân tích các chỉ số định lượng:
- **Precision@5 (Trung bình: 0.6333 / 63.33%)**:
  - Với đa số các truy vấn, hệ thống tìm kiếm TF-IDF có từ 3 đến 4 tài liệu trong top 5 là thực sự liên quan đến chủ đề của truy vấn.
  - Một số tài liệu không liên quan vẫn lọt vào top 5 do chứa các từ khóa đơn lẻ có tần suất hoặc trọng số IDF nhất định trong văn bản, dù toàn bộ ngữ cảnh không thực sự nói về chủ đề tìm kiếm (ví dụ từ *"image"* trong tiêu đề nội thất ghế, hoặc từ *"learning"* trong diễn đàn chia sẻ sở thích cá nhân).
- **Recall@5 (Trung bình: 0.8750 / 87.50%)**:
  - Recall đạt mức cao, phần lớn các tài liệu liên quan trong tập dữ liệu đều được hệ thống phát hiện và xếp vào top 5 kết quả đầu tiên.
- **Mean Reciprocal Rank - MRR (1.0000)**:
  - Đối với toàn bộ 6 query thử nghiệm, tài liệu liên quan xuất hiện ngay tại **Rank 1** ($r_q = 1 \implies RR = 1.0$).
  - Điều này cho thấy với các query có từ khóa rõ ràng, hàm đo độ tương đồng Cosine kết hợp trọng số TF-IDF xếp hạng văn bản chính xác nhất lên ngay đầu danh sách trả về, mang lại trải nghiệm tìm kiếm tốt cho người dùng.

---

### 2. Đối chiếu với dự đoán tại Part C (Prediction 3 — Search):
- Kết quả thực nghiệm hoàn toàn **nhất quán với các dự đoán đã nêu trong Part C**:
  1. **Khớp từ vựng (Lexical Match)**: TF-IDF hoạt động rất hiệu quả khi query chứa các từ vựng đặc trưng, có giá trị IDF cao (như `transformer`, `healthcare`, `classification`, `sql`, `warming`). Những từ này đóng vai trò quyết định giúp văn bản liên quan có điểm cosine similarity vượt trội.
  2. **Vấn đề từ đa nghĩa và ngữ cảnh**: Một số tài liệu dù chứa từ trong query nhưng ngữ cảnh sử dụng lại hoàn toàn khác (ví dụ: query *"medical image classification"* tìm thấy văn bản nói về *"pottery barn stools inside bar"* chỉ vì có từ *"image"* trong tên file ảnh, hoặc query *"database sql query"* xuất hiện văn bản hỏi về vị trí lưu mail filesystem). Điều này minh chứng TF-IDF chỉ tính toán sự trùng khớp từ vựng bề mặt mà không nắm bắt được ngữ cảnh và ngữ nghĩa sâu của câu lệnh truy vấn.


# 12. Part I — Error Analysis

Phần này tiến hành phân tích sâu các lỗi truy vấn (Error Analysis) của hệ thống tìm kiếm TF-IDF từ kết quả thực nghiệm ở Part H, nhằm hiểu rõ cơ chế hoạt động thực tế, điểm mạnh và các giới hạn cốt lõi của phương pháp dựa trên từ vựng (lexical matching).

---

## 1. Hai Queries có kết quả tốt (Good Results)

### Query 1: `"transformer language model"`
- **Expected relevant documents**: `[350, 582, 700, 701]`
- **Retrieved documents**:
  1. `Doc 701` (Score: 0.366049) — **Relevant** *(Chương trình giảng dạy tiếng Pháp ngoại ngữ)*
  2. `Doc 700` (Score: 0.244997) — **Relevant** *(Thảo luận về cấu trúc ngữ nghĩa ngôn ngữ lập trình)*
  3. `Doc 350` (Score: 0.211364) — **Relevant** *(Khoa Ngôn ngữ và Dịch thuật DLT)*
  4. `Doc 582` (Score: 0.162866) — **Relevant** *(Tính năng đa ngôn ngữ trong phần mềm iScripts)*
  5. `Doc 914` (Score: 0.149923) — Non-relevant *(Đầu tư cổ phiếu thị trường phần mềm SQLC)*
- **Đánh giá metric**: $P@5 = 0.8000$ (4/5), $R@5 = 1.0000$ (4/4), $RR = 1.0000$ (Rank 1).

#### Analysis:
1. **Vì sao document đứng đầu?**: `Doc 701` đứng đầu vì chứa từ khóa `"language"` với tần suất xuất hiện cao (high TF) trong một đoạn văn ngắn về giảng dạy ngôn ngữ Pháp, làm tăng mạnh giá trị $TF \times IDF$.
2. **Những từ nào đóng góp nhiều vào similarity?**: Từ `"language"` đóng góp **100%** vào điểm tương đồng Cosine ($0.3660$), do `"language"` có trọng số $IDF \approx 3.540$ tương đối cao trong từ điển. Các từ `"transformer"` và `"model"` không xuất hiện trong tài liệu này.
3. **Có lexical overlap không?**: **Có**. Có sự trùng khớp từ vựng chính xác trên token `"language"`.
4. **Có relevant document nào bị bỏ sót không?**: **Không**. Toàn bộ 4 tài liệu liên quan trong tập ground-truth (`701, 700, 350, 582`) đều được hệ thống đưa vào top 4 kết quả ($R@5 = 100\%$).
5. **Nguồn gốc thành công / hạn chế nhỏ**:
   - **Thành công nhờ Lexical Matching & IDF**: Giúp gom trúng tất cả các tài liệu chứa từ khóa chủ đạo `"language"`.
   - **Hạn chế nhỏ ở Doc 914 (Rank 5)**: `Doc 914` là văn bản về đầu tư tài chính SQLC, lọt vào top 5 chỉ vì chứa từ `"model"` trong ngữ cảnh kinh doanh ("business model"). Đây là sự nhập nhằng do đa nghĩa (polysemy) trong lexical matching.

---

### Query 2: `"natural language processing"`
- **Expected relevant documents**: `[350, 582, 700, 701]`
- **Retrieved documents**:
  1. `Doc 701` (Score: 0.286270) — **Relevant**
  2. `Doc 700` (Score: 0.191600) — **Relevant**
  3. `Doc 350` (Score: 0.165297) — **Relevant**
  4. `Doc 565` (Score: 0.158007) — Non-relevant *(Bản phát hành sửa lỗi bảo trì phần mềm)*
  5. `Doc 582` (Score: 0.127370) — **Relevant**
- **Đánh giá metric**: $P@5 = 0.8000$ (4/5), $R@5 = 1.0000$ (4/4), $RR = 1.0000$ (Rank 1).

#### Analysis:
1. **Vì sao document đứng đầu?**: Tương tự Query 1, `Doc 701` đứng đầu bảng nhờ tần suất xuất hiện dày đặc của từ `"language"`.
2. **Những từ nào đóng góp nhiều vào similarity?**: Từ `"language"` tiếp tục là nhân tố quyết định chính, đóng góp toàn bộ vào dot product giữa query và tài liệu.
3. **Có lexical overlap không?**: **Có**. Trùng khớp từ vựng rõ ràng ở từ `"language"`.
4. **Có relevant document nào bị bỏ sót không?**: **Không**. Đạt Recall tuyệt đối ($R@5 = 1.0000$), cả 4 tài liệu mục tiêu đều nằm trong top 5.
5. **Nguồn gốc failure ở Rank 4 (Doc 565)**:
   - `Doc 565` là bản tin cập nhật bảo trì phần mềm (release notes, bug fixes), lọt vào top 5 vì chứa từ `"natural"` ("natural progression").
   - Nguyên nhân: **Lexical Matching độc lập từng từ** của TF-IDF (Bag-of-Words). Hệ thống xem các từ trong query là rời rạc, không nhận diện được cụm danh từ cố định (noun phrase / collocation) `"natural language processing"`.

---

## 2. Hai Queries có kết quả kém (Poor Results)

### Query 3: `"climate change global warming"`
- **Expected relevant documents**: `[80, 896]`
- **Retrieved documents**:
  1. `Doc 896` (Score: 0.509262) — **Relevant** *(Toàn cầu hóa và tác động biến đổi khí hậu toàn cầu)*
  2. `Doc 291` (Score: 0.148224) — Non-relevant (False Positive) *(Vận động kiểm phiếu bầu cử Đảng Xanh)*
  3. `Doc 80` (Score: 0.116954) — **Relevant** *(Lịch sử điều kiện sống toàn cầu)*
  4. `Doc 110` (Score: 0.099759) — Non-relevant (False Positive) *(Thể chế liên bang và siêu nhà nước châu Âu)*
  5. `Doc 757` (Score: 0.099657) — Non-relevant (False Positive) *(Phần mềm trực quan hóa dữ liệu JMP)*
- **Đánh giá metric**: $P@5 = 0.4000$ (chỉ 2/5 tài liệu đúng), $R@5 = 1.0000$ (2/2), $RR = 1.0000$.

#### Analysis:
1. **Vì sao document đứng đầu?**: `Doc 896` đứng đầu với điểm tương đồng áp đảo ($0.5093$) vì chứa **cả 4 từ** của câu truy vấn: `"warming"`, `"climate"`, `"global"`, và `"change"`.
2. **Những từ nào đóng góp nhiều vào similarity?**:
   - Ở `Doc 896`: Từ `"warming"` ($IDF = 5.116$) đóng góp tới $0.3599$ ($70.7\%$ tổng điểm), `"climate"` ($IDF = 4.510$) đóng góp $0.0799$ ($15.7\%$), `"global"` đóng góp $0.0449$, và `"change"` đóng góp $0.0246$.
   - Ở các tài liệu sai (`Doc 291`, `Doc 110`, `Doc 757`): Mỗi tài liệu chỉ chứa đúng **1 từ đơn lẻ** (ví dụ `Doc 291` chỉ chứa từ `"climate"` trong cuộc vận động của Đảng Xanh, `Doc 110` và `Doc 757` chỉ chứa từ `"global"`).
3. **Có lexical overlap không?**: **Có**, nhưng là lexical overlap một phần (partial overlap) trên các từ đơn lẻ có IDF cao.
4. **Có relevant document nào bị bỏ sót không?**: **Không** bị bỏ sót trong top 5 ($R@5 = 1.0$), nhưng Precision rất thấp ($40\%$) vì có tới **3 False Positives**.
5. **Nguồn gốc của failure**:
   - **Lexical Matching & Giả định Bag-of-Words**: TF-IDF không có cơ chế liên kết cụm từ ("climate change", "global warming"). Chỉ cần một từ trong query xuất hiện trong tài liệu là văn bản đó đã nhận được điểm số đáng kể, làm loãng kết quả tìm kiếm với các tài liệu về chính trị hoặc phần mềm đồ họa.

---

### Query 4: `"database sql query"`
- **Expected relevant documents**: `[408, 757, 768, 914]`
- **Retrieved documents**:
  1. `Doc 408` (Score: 0.226897) — **Relevant** *(Cấu hình extension PHP PDO MSSQL database)*
  2. `Doc 686` (Score: 0.210041) — Non-relevant (False Positive) *(Giải câu đố ô chữ crossword clue)*
  3. `Doc 757` (Score: 0.146647) — **Relevant** *(Cơ sở dữ liệu toàn cầu JMP)*
  4. `Doc 287` (Score: 0.133590) — Non-relevant (False Positive) *(Bản tin xét nghiệm y tế di truyền UnitedHealthcare)*
  5. `Doc 768` (Score: 0.111850) — **Relevant** *(Lưu trữ mail server trên hệ thống tệp)*
- **Đánh giá metric**: $P@5 = 0.6000$ (3/5), $R@5 = 0.7500$ (3/4 - bỏ sót Doc 914), $RR = 1.0000$.

#### Analysis:
1. **Vì sao document đứng đầu?**: `Doc 408` đứng đầu vì chứa các định danh cấu hình mở rộng PHP cho hệ cơ sở dữ liệu Microsoft SQL Server (`"database"` và `"sql"`).
2. **Những từ nào đóng góp nhiều vào similarity?**:
   - Từ `"database"` ($IDF = 4.200$) và `"sql"` ($IDF = 5.809$) đóng góp phần lớn điểm số.
   - Tại `Doc 686` (Rank 2 - False Positive nặng nhất): Chỉ duy nhất từ `"database"` đóng góp toàn bộ $0.2100$ điểm.
3. **Có lexical overlap không?**: **Có**. Tuy nhiên `Doc 686` là trang web giải câu đố ô chữ (Crossword Puzzle), câu văn thực tế là: *"we will try to find the right answer in our crossword clue database"*.
4. **Có relevant document nào bị bỏ sót không?**: **Có**, `Doc 914` (phần mềm SQLC) bị đẩy ra ngoài top 5 ($R@5$ chỉ đạt $75\%$).
5. **Nguồn gốc của failure**:
   - **Đa nghĩa và Ngữ cảnh (Polysemy / Context blindness)**: `Doc 686` hoàn toàn không liên quan đến lập trình hay hệ cơ sở dữ liệu SQL, nhưng từ `"database"` có IDF cao nên kéo toàn bộ điểm của văn bản giải đố ô chữ này lên tận **Rank 2**.
   - **Vocabulary Mismatch**: `Doc 914` nói về công cụ SQLC và lưu trữ dữ liệu nhưng sử dụng ít từ lặp lại chính xác so với câu truy vấn nên bị điểm thấp hơn tài liệu ô chữ.

---

## 3. Failure Case quan trọng nhất: Vocabulary Mismatch Problem (Sự bất đồng về từ vựng)

### Kịch bản thực nghiệm:
Xét trường hợp tìm kiếm y khoa thực tế trong tập dữ liệu:
- **Query**: `"myocardial infarction prevention"` *(Phòng ngừa nhồi máu cơ tim)*
- **Tài liệu mục tiêu (Expected Ground-Truth)**: `Doc 799`
  > *"Can You Have A Heart Attack With A Pacemaker? A pacemaker is not a cure. It will not prevent or stop heart disease, nor will it prevent heart attacks. However, we offer you the best monitoring services to lessen the chance of heart attacks..."*

### Kết quả hệ thống TF-IDF:
- Điểm tương đồng của `Doc 799`: **$0.000000$**
- Vị trí xếp hạng của `Doc 799`: **Không được xếp hạng (Rank $\infty$)**
- Danh sách trả về top 3:
  1. `Doc 35` (Score: 0.1403) — Trang danh bạ website tổng hợp
  2. `Doc 540` (Score: 0.1118) — Bài phỏng vấn nữ luật sư đối tác
  3. `Doc 244` (Score: 0.0243) — Tin tức bóng bầu dục Rugby World Cup

### Giải thích nguyên nhân cốt lõi:
1. **Về mặt ngữ nghĩa y khoa (Semantics)**:
   - Hai thuật ngữ **"heart attack"** *(cơn đau tim)* và **"myocardial infarction"** *(nhồi máu cơ tim)* là **từ đồng nghĩa hoàn toàn (exact medical synonyms)** cùng chỉ một hiện tượng bệnh lý tắc nghẽn động mạch vành hoại tử cơ tim.
   - Con người lập tức hiểu rằng `Doc 799` là tài liệu cực kỳ giá trị và trực tiếp trả lời câu hỏi phòng ngừa nhồi máu cơ tim.
2. **Về mặt toán học của TF-IDF (Lexical Space)**:
   - Vector của Query $\vec{q}$ chỉ có các tọa độ khác 0 tại các chiều: `['myocardial', 'infarction', 'prevention']`.
   - Vector của `Doc 799` $\vec{d}_{799}$ chỉ có các tọa độ khác 0 tại: `['heart', 'attack', 'pacemaker', 'disease', ...]`.
   - Giao của tập từ vựng bệnh lý: $\text{Query} \cap \text{Doc 799} = \emptyset$.
   - Tích vô hướng:
     $$\vec{q} \cdot \vec{d}_{799} = \sum_{w \in V} q_w \cdot d_w = 0 \implies \text{Cosine Similarity} = 0$$
   - Hai vector hoàn toàn **trực giao (orthogonal)** trong không gian vector nhiều chiều của TF-IDF.
3. **Hậu quả**:
   - `Doc 799` nhận điểm 0 và bị bỏ sót hoàn toàn ($Recall = 0$).
   - Trong khi đó, các tài liệu không liên quan (`Doc 35`, `Doc 540`) chỉ tình cờ chứa từ phụ `"prevention"` lại nhận điểm cao và chiếm trọn top 1, top 2.
   - Đây chính là **Vocabulary Mismatch Problem** — hạn chế lớn nhất và không thể khắc phục được của mô hình TF-IDF truyền thống.


In [38]:
# Thực nghiệm minh họa Failure Case quan trọng: Vocabulary Mismatch Problem
query_failure = "myocardial infarction prevention"
qv = build_query_vector(query_failure, vocab, idf_list)

print(f"QUERY: '{query_failure}'")
print("-" * 70)

# Kiểm tra điểm số của Doc 799 (văn bản về Heart Attack)
dv_799 = tfidf_matrix[799]
score_799 = cosine_similarity(qv, dv_799)
print(f"Doc 799 (Chủ đề: 'Heart Attack'):")
print(f"  Snippet: \"{documents[799][:120].strip()}...\"")
print(f"  Similarity Score: {score_799:.6f}  <-- HOÀN TOÀN BẰNG 0!\\n")

# Top 3 kết quả mà TF-IDF thực sự trả về cho query này:
search_res_failure = search_documents(query_failure, documents, vocab, idf_list, tfidf_matrix, top_k=3)
print("Top 3 kết quả TF-IDF thực tế trả về (False Positives):")
for rank, (doc_id, score) in enumerate(search_res_failure, start=1):
    preview = documents[doc_id].replace("\n", " ").strip()[:90]
    print(f"  Rank {rank}: Doc {doc_id} (Score: {score:.6f}) -> \"{preview}...\"")


QUERY: 'myocardial infarction prevention'
----------------------------------------------------------------------
Doc 799 (Chủ đề: 'Heart Attack'):
  Snippet: "can you have a heart attack with a pacemaker? a pacemaker is not a cure. it will not prevent or stop heart disease, nor..."
  Similarity Score: 0.000000  <-- HOÀN TOÀN BẰNG 0!

Top 3 kết quả TF-IDF thực tế trả về (False Positives):
  Rank 1: Doc 35 (Score: 0.140264) -> "currently there are 3 websites listed for this state. we are committed to providing for th..."
  Rank 2: Doc 540 (Score: 0.111828) -> "arameh zargham o’boyle, of mintz levin, is the first female partner in the growing los ang..."
  Rank 3: Doc 244 (Score: 0.024338) -> "sign up to lineout to stay up to date with all of the latest rupa news / thanks for subscr..."


# 13. Part J — From Failure to the Next NLP Representation

## 1. Hạn chế cốt lõi của mô hình TF-IDF

Mô hình TF-IDF và không gian vector truyền thống (Vector Space Model) biểu diễn văn bản dựa trên **thống kê từ vựng bề mặt (Lexical Statistics)**. Cách tiếp cận này bộc lộ những điểm yếu chí mạng:

1. **Giả định tính trực giao (Orthogonality Assumption)**:
   - Mỗi từ trong từ điển $V$ được coi là một chiều độc lập trong không gian $|V|$ chiều.
   - Khoảng cách giữa hai từ bất kỳ (dù đồng nghĩa như `"heart attack"` và `"myocardial infarction"`, hay không liên quan như `"cat"` và `"refrigerator"`) đều như nhau: tích vô hướng bằng 0.
2. **Vấn đề từ đồng nghĩa (Synonymy / Vocabulary Mismatch)**:
   - Người dùng và tác giả văn bản thường sử dụng các từ vựng khác nhau để diễn đạt cùng một khái niệm. TF-IDF không có khả năng nhận ra sự tương đồng này nếu không trùng khớp chính xác mặt chữ.
3. **Vấn đề từ đa nghĩa (Polysemy) và Mất ngữ cảnh**:
   - Từ `"bank"` trong *"river bank"* (bờ sông) và *"bank account"* (tài khoản ngân hàng) đều có cùng một tọa độ và cùng một giá trị IDF.
   - Bỏ qua trật tự từ (Bag-of-Words): câu *"dog bites man"* và *"man bites dog"* có biểu diễn vector hoàn toàn giống nhau dù ý nghĩa trái ngược.

---

## 2. Câu hỏi cuối buổi: Biểu diễn Similarity về Nghĩa thay vì Similarity về Từ

> **Câu hỏi**: *Làm thế nào để biểu diễn được similarity về nghĩa (semantic similarity) thay vì chỉ similarity về từ (lexical similarity)?*

**Trả lời**:
- Các từ có ngữ nghĩa tương đồng (như *"heart attack"* và *"infarction"*) thường xuất hiện chung với những từ ngữ cảnh tương tự (như *"hospital"*, *"patient"*, *"doctor"*, *"chest pain"*, *"treatment"*).
- Thay vì biểu diễn từ dưới dạng vector thưa one-hot trực giao, ta ánh xạ các từ vào một **không gian vector liên tục, dày đặc (Dense Vector Space $\mathbb{R}^d$ với $d \ll |V|$, thường từ 100 đến 1024 chiều)**.
- Trong không gian này, các từ hoặc văn bản có sự tương đồng về ngữ cảnh sẽ có vị trí hình học gần nhau, thể hiện qua điểm **Cosine Similarity cao** ngay cả khi không trùng lặp mặt chữ.
